# 16-тарау. Мағыналық іздеу: векторлар мен эмбеддингтер

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 16.1. Сөзбен іздеу және мағынамен іздеу

In [ ]:
import re
import urllib.request
import numpy as np

RAW = ("https://raw.githubusercontent.com/amirtemir1996n3-dotcom/"
       "mashina-qalai-uirenedi/main/")
for path in ["data/qa.txt", "data/kitap.txt"]:
    urllib.request.urlretrieve(RAW + path, path.split("/")[-1])

terms, docs = [], []
for pair in open("qa.txt", encoding="utf-8").read().strip().split("\n\n"):
    question, answer = pair.split("\n")
    term = question[len("Сұрақ: "):-len(" деген не?")]
    terms.append(term)
    docs.append(term + ": " + answer[len("Жауап: "):])
print(len(docs), "құжат")
print(docs[33])

## 16.2. Екі вектор қаншалықты ұқсас?

In [ ]:
def cosine(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

a = np.array([3.0, 1.0])
print(f"{cosine(a, 2 * a):.2f}  {cosine(a, np.array([-1.0, 3.0])):.2f}")

## 16.3. Сөздер қапшығы және TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

word_vec = TfidfVectorizer()
D = word_vec.fit_transform(docs)
print("матрица:", D.shape)
idf = dict(zip(word_vec.get_feature_names_out(), word_vec.idf_))
for w in ["модельдің", "деректер", "сүзгі", "жаттап"]:
    print(f"{w:10} idf = {idf[w]:.2f}")

In [ ]:
def search(query, vec, D, k=3):
    q = vec.transform([query])
    scores = (D @ q.T).toarray().ravel()
    best = np.argsort(-scores)[:k]
    return [(terms[i], round(float(scores[i]), 2)) for i in best]

print(search("Желінің ең кішкентай есептеу бірлігі", word_vec, D))

## 16.4. Жалғаулар мәселесі

In [ ]:
query = "Сүзгілер суретті бойлай жылжиды"
print(search(query, word_vec, D))

In [ ]:
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
Dc = char_vec.fit_transform(docs)
print("матрица:", Dc.shape)
grams = char_vec.build_analyzer()
print(sorted(set(grams("сүзгілер")) & set(grams("сүзгіні"))))
print(search(query, char_vec, Dc))

In [ ]:
query = "Чат-бот жоқ фактіні сеніммен айтты"
print(search(query, word_vec, D))
print(search(query, char_vec, Dc))

## 16.5. Сынақ: 16 қиын сұрақ

In [ ]:
hard = [
    ("Алаяқтар туысыңның дауысымен қоңырау шалып, ақша сұрады", "Дипфейк"),
    ("Ертеңгі ауа райын болжау", "Уақыттық қатар"),
    ("Тауда ең тік жолмен төмен қарай түсу", "Градиенттік түсу"),
    ("Суреттен жиектер мен бұрыштарды табатын амал", "Конволюция"),
    ("Сөйлемдегі әр сөз қай сөздерге қарау керегін өзі шешеді",
     "Назар аудару механизмі"),
    ("Дүкен сатып алушыларын ұқсастығы бойынша топтау", "Кластерлеу"),
    ("Бір мысықтың суретін бұрып, жаңа суреттер алу", "Деректерді кеңейту"),
    ("Пәтердің бағасын болжау", "Регрессия"),
    ("Хат спам ба, әлде жоқ па?", "Жіктеу"),
    ("Кешегі сатылымға қарап, бүгінгісін болжау", "Кешігу белгісі"),
    ("Миллиондаған суретте үйренген желіні рентгенге қолдану",
     "Трансферлік оқыту"),
    ("Сұрақ-жауап боты жоқ кітапты ойдан шығарды", "Галлюцинация"),
    ("Шахмат ойнайтын, сурет салатын ақылды бағдарлама",
     "Жасанды интеллект (ЖИ)"),
    ("Қателік қай салмақтан келгенін соңынан басына қарай есептеу",
     "Кері таралу"),
    ("Ойыншылардың бойы мен салмағын бірдей шкалаға келтіру",
     "Нормализация"),
    ("Неше рет қатені азайтып, қадам жасаймыз — сол қадамның өлшемі",
     "Оқыту қарқыны"),
]

In [ ]:
def evaluate(embed, queries=hard):
    E = embed(docs)
    top1 = top3 = 0
    for q, term in queries:
        scores = E @ embed([q])[0]
        best = [terms[i] for i in np.argsort(-scores)[:3]]
        top1 += best[0] == term
        top3 += term in best
    return top1, top3

def report():
    for name, (t1, t3) in results.items():
        print(f"{name:16} бірінші орында: {t1:2}/16   үштікте: {t3:2}/16")

dense = lambda vec: (lambda texts: vec.transform(texts).toarray())
results = {"сөз TF-IDF": evaluate(dense(word_vec)),
           "n-грамм TF-IDF": evaluate(dense(char_vec))}
report()

## 16.6. Мағына көршілерде жасырынған

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

book = [p for p in open("kitap.txt", encoding="utf-8").read().split("\n")
        if len(p) > 40]
lsa_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
X = lsa_vec.fit_transform(book + docs)
svd = TruncatedSVD(100, random_state=0).fit(X)
print("абзацтар:", len(book), "  матрица:", X.shape)

def lsa(texts):
    return normalize(svd.transform(lsa_vec.transform(texts)))

results["LSA"] = evaluate(lsa)
report()

In [ ]:
from collections import Counter

counts = Counter(re.findall(r"[а-яәіңғүұқөһё]+", " ".join(book).lower()))
vocab = [w for w, c in counts.items() if c >= 3 and len(w) > 3]
V = lsa(vocab)

def neighbours(word, k=5):
    sims = V @ lsa([word])[0]
    near = [vocab[i] for i in np.argsort(-sims) if vocab[i][:4] != word[:4]]
    return ", ".join(near[:k])

for w in ["градиент", "дипфейк", "спам", "оқушы", "температура", "сурет"]:
    print(f"{w:12} {neighbours(w)}")

## 16.7. Эмбеддинг моделін өзіміз үйретеміз

In [ ]:
import random
import torch
import torch.nn.functional as F

def sentences(p):
    return [s for s in re.split(r"(?<=[.!?])\s+", p) if len(s) > 20]

paras = [p for p in book if len(sentences(p)) >= 2]

def make_batch(chunk):
    queries, passages = [], []
    for p in chunk:
        s = sentences(p)
        j = random.randrange(len(s))
        queries.append(s[j])
        passages.append(" ".join(s[:j] + s[j + 1:]))
    return queries, passages

random.seed(0)
q, p = make_batch(paras[:1])
print(len(paras), "абзац\nСҰРАҚ:", q[0], "\nЖАУАП:", p[0][:150], "…")

In [ ]:
import zlib
import torch.nn as nn

BUCKETS = 2 ** 18

def ngram_ids(text):
    ids = []
    for w in re.findall(r"\w+", text.lower()):
        w = f"<{w}>"
        for n in (3, 4, 5):
            ids += [zlib.crc32(w[i:i + n].encode()) % BUCKETS
                    for i in range(len(w) - n + 1)]
    return ids

class Encoder(nn.Module):
    def __init__(self, dim=256):
        super().__init__()
        self.emb = nn.EmbeddingBag(BUCKETS, dim, mode="mean")
        nn.init.normal_(self.emb.weight, std=0.1)

    def forward(self, texts):
        flat, offsets = [], []
        for t in texts:
            offsets.append(len(flat))
            flat += ngram_ids(t)
        v = self.emb(torch.tensor(flat), torch.tensor(offsets))
        return F.normalize(v, dim=-1)

torch.manual_seed(0)
enc = Encoder()
embed = lambda texts: enc(texts).detach().numpy()
print("параметрлер:", sum(p.numel() for p in enc.parameters()))
print("оқытуға дейін:", evaluate(embed))

In [ ]:
opt = torch.optim.Adam(enc.parameters(), lr=0.02)
for epoch in range(6):
    random.shuffle(paras)
    for i in range(0, len(paras) - 63, 64):
        q, p = make_batch(paras[i:i + 64])
        logits = enc(q) @ enc(p).T / 0.05
        loss = F.cross_entropy(logits, torch.arange(len(q)))
        opt.zero_grad()
        loss.backward()
        opt.step()
    print(f"{epoch + 1}-дәуір: шығын {loss.item():.3f}, "
          f"сынақ {evaluate(embed)}")

results["біздің модель"] = evaluate(embed)
report()

## 16.8. Нағыз эмбеддинг модельдері

**Міндетті емес ұяшық.** Бұл код қосымша бағдарламаны, интернеттегі қызметті немесе өз файлыңызды қажет етеді (толығырақ кітапта). Іске қосу үшін ұяшықтың бірінші жолын (`%%script echo ...`) өшіріңіз.

In [ ]:
%%script echo "Міндетті емес ұяшық: іске қосу үшін осы бірінші жолды өшіріңіз."
# pip install sentence-transformers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-base")

def e5(texts):
    prefix = "passage: " if texts is docs else "query: "
    return model.encode([prefix + t for t in texts],
                        normalize_embeddings=True)

results["multilingual-e5"] = evaluate(e5)
report()

## 16.9. Миллион векторда іздеу

In [ ]:
from sklearn.cluster import KMeans

rng = np.random.default_rng(0)
N, d = 50_000, 64
centers = rng.normal(size=(500, d))

def sample(n):
    v = centers[rng.integers(0, 500, n)] + rng.normal(size=(n, d))
    return v / np.linalg.norm(v, axis=1, keepdims=True)

X, Q = sample(N), sample(200)
exact = np.argsort(-(Q @ X.T), axis=1)[:, :10]      # дәл жауап

km = KMeans(n_clusters=256, n_init=1, random_state=0).fit(X)
lists = [np.where(km.labels_ == c)[0] for c in range(256)]

def ivf_search(q, nprobe, k=10):
    near = np.argsort(-(km.cluster_centers_ @ q))[:nprobe]
    cand = np.concatenate([lists[c] for c in near])
    return cand[np.argsort(-(X[cand] @ q))[:k]], len(cand)

ivf_rows = []
for nprobe in [1, 2, 4, 8, 16, 32]:
    recall = scanned = 0
    for q, ex in zip(Q, exact):
        found, n = ivf_search(q, nprobe)
        recall += len(set(found) & set(ex)) / 10
        scanned += n
    ivf_rows.append((nprobe, recall / len(Q), scanned / len(Q) / N))
    print(f"nprobe={nprobe:2}: толықтық {recall / len(Q):.2f}, "
          f"тексерілген векторлар {scanned / len(Q) / N:.1%}")